# Reducción gravimétrica – Facatativá

**Taller 2 de Geofísica – Gravimetría** · Universidad Distrital Francisco José de Caldas
Autores: Samuel Leandro Calderón Pachón (20221025019) · Daniel Ernesto Chicaeme Sánchez (20222025077)
Profesor: Andrés Cárdenas Contreras

Este cuaderno reproduce toda la reducción de los datos de campo, paso a paso, a partir de los Excel originales del profesor.

**Cómo usarlo**
1. Abrir este cuaderno en VS Code (extensión *Jupyter*) y elegir el kernel de Python (arriba a la derecha).
2. Ejecutar las celdas **en orden**, de arriba hacia abajo (Shift + Enter o *Run All*).
3. La primera celda instala las librerías; solo hace falta una vez por computador.

**Estructura esperada del repositorio**
```
datos/originales/   ← los 4 Excel del profesor (sin modificar)
datos/procesados/   ← los Excel que genera este cuaderno
scripts/            ← este cuaderno
```

## 0. Instalación de librerías (una sola vez por computador)

In [1]:
%pip install pyproj openpyxl numpy pandas
%pip install tidegravity --no-deps

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Daniel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Daniel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Configuración general
Ubica la carpeta del repositorio, define las rutas y los parámetros comunes del proyecto.

In [2]:
import datetime
from pathlib import Path

import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from pyproj import Transformer
import tidegravity as tg

# Busca la raíz del repositorio (la carpeta que contiene datos/originales)
raiz = Path.cwd()
while not (raiz / "datos" / "originales").exists():
    if raiz.parent == raiz:
        raise SystemExit("No encontré la carpeta datos/originales. Abre el cuaderno dentro del repositorio.")
    raiz = raiz.parent

ORIG = raiz / "datos" / "originales"
PROC = raiz / "datos" / "procesados"
PROC.mkdir(parents=True, exist_ok=True)

ARCHIVO_CAMPO  = ORIG / "DATOS GRAVIMETRICOS CAMPO.xlsx"
ARCHIVO_AMARRE = ORIG / "AMARRE BASE.xlsx"
PASO1 = PROC / "Paso1_Datos_Organizados_Facatativa.xlsx"
PASO2 = PROC / "Paso2_Coordenadas_Facatativa.xlsx"
PASO45 = PROC / "Paso4y5_Mareas_Facatativa.xlsx"

# Parámetros del proyecto (indicados por el profesor, clase 28 oct 2026)
CONST_GRAV1 = 0.0959                      # ciclos 1 a 7
CONST_GRAV2 = 0.0433                      # ciclos 8 a 11
FECHA_GRAV1 = datetime.datetime(2010, 8, 10)
FECHA_GRAV2 = datetime.datetime(2010, 8, 11)
UTC_OFFSET = 5                            # hora Colombia = UTC − 5
EPSG_SALIDA = "EPSG:3116"                 # MAGNA-SIRGAS / Colombia Bogotá zone (Gauss-Krüger origen Bogotá)
BASE = "GPS-3"

# Estilos para los Excel
FUENTE = "Arial"
f = lambda **k: Font(name=FUENTE, size=10, **k)
ENC_FONT = Font(name=FUENTE, size=10, bold=True, color="FFFFFF")
ENC_FILL = PatternFill("solid", start_color="1F3864")
AMARILLO = PatternFill("solid", start_color="FFFF00")
AZUL = PatternFill("solid", start_color="DDEBF7")
NARANJA = PatternFill("solid", start_color="FCE4D6")
BORDE = Border(*(Side(style="thin", color="BFBFBF"),) * 4)

def encabezado(celda, texto):
    celda.value, celda.font, celda.fill, celda.border = texto, ENC_FONT, ENC_FILL, BORDE
    celda.alignment = Alignment(wrap_text=True, horizontal="center", vertical="center")

print("Repositorio:", raiz)

Repositorio: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa


## Paso 1. Organizar y depurar los datos

1. Se lee `DATOS GRAVIMETRICOS CAMPO.xlsx` y se separan los **ciclos**: cada ciclo abre y cierra en la base GPS-3.
2. A cada fila se le da un **ID único** (`Cxx-Estación`), porque los nombres de estación se repiten entre ciclos pero son puntos distintos.
3. Se aplican las **reglas del profesor** (clase 28 oct 2026):
   - **Regla 1:** si una estación tiene una hora inferior a la esperada dentro del ciclo, solamente se cambia de posición (se ordena por hora).
   - **Regla 2:** si una estación tiene una hora inferior a la hora de inicio en la base, se elimina.
4. Se genera `Paso1_Datos_Organizados_Facatativa.xlsx` con una hoja por ciclo, una hoja **Resumen** y una hoja **Leyenda**.

In [3]:
# --- Leer los ciclos del archivo de campo ---
hoja = openpyxl.load_workbook(ARCHIVO_CAMPO).active
ciclos, actual = [], None
for fila, r in enumerate(hoja.iter_rows(values_only=True), start=1):
    if r[0] is None or str(r[0]).lower().startswith("estac"):
        continue                                   # filas vacías o de encabezado
    dato = dict(fila=fila, est=r[0], lat=r[1], lon=r[2], h=r[3], t=r[4], c=r[5], L=r[6])
    if r[0] == BASE:
        if actual is None:
            actual = [dato]                        # apertura del ciclo
        else:
            actual.append(dato); ciclos.append(actual); actual = None   # cierre del ciclo
    elif actual is not None:
        actual.append(dato)
print("Ciclos encontrados:", len(ciclos))

# --- Reglas del profesor ---
movidas, eliminadas = [], []
for n, ciclo in enumerate(ciclos, start=1):
    apertura, cierre, estaciones = ciclo[0], ciclo[-1], ciclo[1:-1]
    # Regla 2: eliminar estaciones con hora menor a la de apertura en la base
    for d in estaciones:
        if d["t"] < apertura["t"]:
            eliminadas.append((n, d["est"], d["t"]))
    estaciones = [d for d in estaciones if d["t"] >= apertura["t"]]
    # Regla 1: ordenar por hora; registrar las estaciones que cambian de posición
    #          (una estación está fuera de lugar si su hora es menor que la de alguna estación anterior)
    hora_max = apertura["t"]
    for d in estaciones:
        if d["t"] < hora_max:
            movidas.append((n, d["est"], d["t"]))
        hora_max = max(hora_max, d["t"])
    ordenadas = sorted(estaciones, key=lambda d: d["t"])
    ciclos[n - 1] = [apertura] + ordenadas + [cierre]

print("Regla 1 – estaciones reordenadas:", movidas or "ninguna")
print("Regla 2 – estaciones eliminadas:", eliminadas or "ninguna")

# --- Construir el Excel del paso 1 ---
wb = openpyxl.Workbook()
ley = wb.active; ley.title = "Leyenda"
lineas = [
    ["Paso 1 – Organización y depuración de datos gravimétricos"],
    ["Proyecto: Determinación de la Anomalía de Bouguer Total en una zona del Municipio de Facatativá"],
    ["Fuente: DATOS GRAVIMETRICOS CAMPO.xlsx (Universidad Nacional – Universidad Distrital, 2010)"],
    [],
    ["Parámetros"],
    ["Constante gravímetro 1", CONST_GRAV1, "Fecha gravímetro 1", FECHA_GRAV1.date(), "Ciclos 1 a 7"],
    ["Constante gravímetro 2", CONST_GRAV2, "Fecha gravímetro 2", FECHA_GRAV2.date(), "Ciclos 8 a 11"],
    [],
    ["Reglas de depuración (clase 28 oct 2026)"],
    ["Regla 1", "Hora inferior a la esperada dentro del ciclo → solo se cambia de posición."],
    ["", "Resultado: " + ("; ".join(f"ciclo {n}, {e} (hora {t:.2f})" for n, e, t in movidas) or "ninguna")],
    ["Regla 2", "Hora inferior a la hora de inicio en la base → se elimina."],
    ["", "Resultado: " + ("; ".join(f"ciclo {n}, {e} (hora {t:.2f})" for n, e, t in eliminadas) or "ninguna estación cumple la condición")],
]
for l in lineas:
    ley.append(l)
for fila in ley.iter_rows():
    for c in fila:
        c.font = f()
ley["A1"].font = Font(name=FUENTE, size=14, bold=True)
for ref in ("B6", "B7", "D6", "D7"):
    ley[ref].fill, ley[ref].font = AMARILLO, f(color="0000FF")
ley["D6"].number_format = ley["D7"].number_format = "dd/mm/yyyy"
for col, ancho in zip("ABCDE", (24, 14, 20, 14, 60)):
    ley.column_dimensions[col].width = ancho

columnas = ["Orden", "ID", "Estación", "Latitud (°)", "Longitud original (°)", "Longitud (° W)",
            "Altura elipsoidal (m)", "Hora decimal (h)", "Hora (hh:mm)", "Constante", "Lectura dial",
            "Revisión regla 1 (orden)", "Revisión regla 2 (inicio base)", "Fila en archivo original", "Observación"]
anchos = [7, 14, 10, 12, 14, 13, 13, 12, 11, 10, 11, 16, 18, 12, 48]
rangos = []
for n, ciclo in enumerate(ciclos, start=1):
    ws = wb.create_sheet(f"Ciclo {n}")
    grav1 = n <= 7
    ws["A1"] = f"Ciclo {n}"; ws["A1"].font = Font(name=FUENTE, size=13, bold=True)
    ws["A2"], ws["B2"] = "Fecha", "=Leyenda!D6" if grav1 else "=Leyenda!D7"
    ws["A3"], ws["B3"] = "Constante", "=Leyenda!B6" if grav1 else "=Leyenda!B7"
    ws["D2"], ws["E2"] = "Base", BASE
    ws["D3"], ws["E3"] = "Gravímetro", 1 if grav1 else 2
    ws["B2"].number_format = "dd/mm/yyyy"
    for j, texto in enumerate(columnas, start=1):
        encabezado(ws.cell(5, j), texto)
        ws.column_dimensions[get_column_letter(j)].width = anchos[j - 1]
    ws.row_dimensions[5].height = 42
    ini = 6; fin = ini + len(ciclo) - 1
    for k, d in enumerate(ciclo):
        r = ini + k
        es_base = k in (0, len(ciclo) - 1)
        obs = ("Apertura" if k == 0 else "Cierre") + f" del ciclo en la base {BASE}" if es_base else ""
        if (n, d["est"], d["t"]) in movidas and not es_base:
            obs = "Regla 1: hora menor a la esperada; se cambió de posición"
        valores = [k + 1, f"C{n:02d}-{d['est']}", d["est"], d["lat"], d["lon"], f"=-E{r}", d["h"], d["t"],
                   f'=TEXT(H{r}/24,"hh:mm")', "=$B$3", d["L"],
                   "OK" if k == 0 else f'=IF(H{r}>=H{r-1},"OK","Fuera de orden")',
                   f'=IF(H{r}>=$H${ini},"OK","Eliminar")', d["fila"], obs]
        for j, v in enumerate(valores, start=1):
            c = ws.cell(r, j, v); c.font = f(); c.border = BORDE
            c.number_format = {4: "0.000000", 5: "0.000000", 6: "0.000000", 7: "0.000", 8: "0.000", 10: "0.0000", 11: "0.0"}.get(j, "General")
            if es_base: c.fill = AZUL
            if obs.startswith("Regla"): c.fill = NARANJA
    ws.freeze_panes = "C6"
    ws.cell(fin + 2, 1, "Estaciones (sin base)").font = f(bold=True); ws.cell(fin + 2, 4, f"=COUNTA(C{ini+1}:C{fin-1})")
    ws.cell(fin + 3, 1, "Filas fuera de orden").font = f(bold=True); ws.cell(fin + 3, 4, f'=COUNTIF(L{ini}:L{fin},"Fuera de orden")')
    ws.cell(fin + 4, 1, "Filas a eliminar").font = f(bold=True); ws.cell(fin + 4, 4, f'=COUNTIF(M{ini}:M{fin},"Eliminar")')
    rangos.append((ini, fin))

res = wb.create_sheet("Resumen", 1)
res["A1"] = "Resumen de ciclos – Paso 1"; res["A1"].font = Font(name=FUENTE, size=13, bold=True)
cols_res = ["Ciclo", "Fecha", "Gravímetro", "Constante", "Hora apertura (h)", "Hora cierre (h)", "Duración (h)",
            "N.º estaciones", "Lectura apertura GPS-3", "Lectura cierre GPS-3", "Diferencia de cierre (dial)",
            "Fuera de orden", "A eliminar"]
for j, texto in enumerate(cols_res, start=1):
    encabezado(res.cell(3, j), texto); res.column_dimensions[get_column_letter(j)].width = 13
for n, (ini, fin) in enumerate(rangos, start=1):
    r, q = 3 + n, f"'Ciclo {n}'!"
    for j, v in enumerate([n, f"={q}B2", f"={q}E3", f"={q}B3", f"={q}H{ini}", f"={q}H{fin}", f"=F{r}-E{r}",
                           f"={q}D{fin+2}", f"={q}K{ini}", f"={q}K{fin}", f"=J{r}-I{r}",
                           f"={q}D{fin+3}", f"={q}D{fin+4}"], start=1):
        c = res.cell(r, j, v); c.font = f(); c.border = BORDE
        c.number_format = {2: "dd/mm/yyyy", 5: "0.000", 6: "0.000", 7: "0.000"}.get(j, "General")

wb.save(PASO1)
print("Guardado:", PASO1)

Ciclos encontrados: 11
Regla 1 – estaciones reordenadas: [(2, 'SP526', 12.2)]
Regla 2 – estaciones eliminadas: ninguna
Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso1_Datos_Organizados_Facatativa.xlsx


## Paso 2. Coordenadas MAGNA-SIRGAS Gauss-Krüger origen Bogotá

Las coordenadas geográficas (WGS84) de cada estación se transforman a coordenadas planas **MAGNA-SIRGAS / Colombia Bogotá zone (EPSG:3116)**, que es el sistema que pide el taller para todos los mapas.
La longitud del archivo original viene en positivo; se usa en negativo porque es longitud Oeste.

**Comprobación:** GPS-3 debe dar Este ≈ 974495.6 m y Norte ≈ 1025740.3 m.

In [4]:
transformador = Transformer.from_crs("EPSG:4326", EPSG_SALIDA, always_xy=True)
wb = openpyxl.load_workbook(PASO1)

for ws in wb.worksheets:
    if not ws.title.startswith("Ciclo"):
        continue
    encabezado(ws["P5"], "Este (m)"); encabezado(ws["Q5"], "Norte (m)")
    ws.column_dimensions["P"].width = ws.column_dimensions["Q"].width = 14
    fila = 6
    while ws[f"D{fila}"].value is not None:
        lat, lon = ws[f"D{fila}"].value, -ws[f"E{fila}"].value
        este, norte = transformador.transform(lon, lat)
        ws[f"P{fila}"], ws[f"Q{fila}"] = round(este, 3), round(norte, 3)
        fila += 1

wb.save(PASO2)
gps3 = wb["Ciclo 1"]
print(f"GPS-3 → Este = {gps3['P6'].value} m, Norte = {gps3['Q6'].value} m")
print("Guardado:", PASO2)

GPS-3 → Este = 974495.614 m, Norte = 1025740.326 m
Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso2_Coordenadas_Facatativa.xlsx


## Paso 3. Amarre a la base (gravedad absoluta de GPS-3)

B24NW1 es un punto de gravedad conocida (**977389.51 mGal**). Con cada gravímetro se midió varias veces de ida y vuelta entre B24NW1 y GPS-3 (6 lecturas → 5 diferencias).

1. Cada lectura se multiplica por la constante de su gravímetro.
2. Se calcula la diferencia promedio B24NW1 − GPS-3 con cada gravímetro y se promedian los dos.
3. g(GPS-3) = g(B24NW1) − diferencia promedio.

El Excel del profesor da 977384.730 mGal porque una de sus celdas tiene la diferencia escrita a mano; la diferencia con este cálculo es de 0.003 mGal.

In [5]:
G_B24NW1 = 977389.51   # mGal, valor conocido (AMARRE BASE.xlsx)

amarre = openpyxl.load_workbook(ARCHIVO_AMARRE)
# Lecturas de dial en las filas 9 a 14 de cada hoja
lect_g1 = [amarre.worksheets[0][f"I{r}"].value for r in range(9, 15)]
lect_g2 = [amarre.worksheets[1][f"K{r}"].value for r in range(9, 15)]

def diferencia_promedio(lecturas, constante):
    g = np.array(lecturas) * constante
    return np.mean(np.abs(np.diff(g)))         # |B24NW1 − GPS-3| entre lecturas consecutivas

d1 = diferencia_promedio(lect_g1, CONST_GRAV1)
d2 = diferencia_promedio(lect_g2, CONST_GRAV2)
d_prom = (d1 + d2) / 2
G_BASE = G_B24NW1 - d_prom

print(f"Diferencia gravímetro 0.0959: {d1:.3f} mGal")
print(f"Diferencia gravímetro 0.0433: {d2:.3f} mGal")
print(f"Diferencia promedio:          {d_prom:.3f} mGal")
print(f"g(GPS-3) = {G_B24NW1} − {d_prom:.3f} = {G_BASE:.3f} mGal")

Diferencia gravímetro 0.0959: 4.762 mGal
Diferencia gravímetro 0.0433: 4.803 mGal
Diferencia promedio:          4.783 mGal
g(GPS-3) = 977389.51 − 4.783 = 977384.727 mGal


## Pasos 4 y 5. Lecturas en mGal y corrección por mareas

**Paso 4:** lectura (mGal) = lectura del dial × constante del gravímetro.

**Paso 5:** el efecto de marea terrestre se calcula con el método de **Longman (1959)** (librería *Tidegravity*), con la latitud, longitud, altura, fecha y hora UTC de cada lectura.
La corrección es el efecto con signo contrario: **corrección = − efecto de marea**.

- Ciclos 1 a 7 (gravímetro 0.0959): 10 de agosto de 2010
- Ciclos 8 a 11 (gravímetro 0.0433): 11 de agosto de 2010
- Hora UTC = hora local + 5 h

**Comprobación (ciclo 3, GPS-3):** lectura 103.6967 mGal, efecto 0.1473 mGal, corrección −0.1473 mGal.

In [6]:
wb = openpyxl.load_workbook(PASO2)

for ws in wb.worksheets:
    if not ws.title.startswith("Ciclo"):
        continue
    ciclo = int(ws.title.split()[1])
    fecha = FECHA_GRAV1 if ciclo <= 7 else FECHA_GRAV2
    for col, texto in (("R", "Lectura (mGal)"), ("S", "Hora UTC"),
                       ("T", "Efecto de marea (mGal)"), ("U", "Corrección por marea (mGal)")):
        encabezado(ws[f"{col}5"], texto); ws.column_dimensions[col].width = 14
    fila = 6
    while ws[f"D{fila}"].value is not None:
        lat, lon = ws[f"D{fila}"].value, -ws[f"E{fila}"].value
        alt, hora = ws[f"G{fila}"].value, ws[f"H{fila}"].value
        t_utc = fecha + datetime.timedelta(hours=hora + UTC_OFFSET)
        gm, gs, g0 = tg.solve_longman_tide_scalar(lat, lon, alt, t_utc)
        ws[f"R{fila}"] = f"=K{fila}*J{fila}"          # Paso 4
        ws[f"S{fila}"] = t_utc.strftime("%Y-%m-%d %H:%M")
        ws[f"T{fila}"] = round(float(g0), 4)           # efecto de marea (Longman)
        ws[f"U{fila}"] = f"=-T{fila}"                  # Paso 5: corrección
        for col in "RTU":
            ws[f"{col}{fila}"].number_format = "0.0000"
        fila += 1

wb.save(PASO45)
c3 = wb["Ciclo 3"]
print(f"Ciclo 3, GPS-3 → efecto de marea = {c3['T6'].value} mGal, corrección = {-c3['T6'].value} mGal")
print("Guardado:", PASO45)

Ciclo 3, GPS-3 → efecto de marea = 0.1473 mGal, corrección = -0.1473 mGal
Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso4y5_Mareas_Facatativa.xlsx


## Paso 6. Deriva instrumental

Con las lecturas repetidas en un mismo punto (`DERIVA.xlsx`) se ajusta una recta **g = D·t + b**. La pendiente **D (mGal/h)** es la deriva del gravímetro.

Reglas del profesor (clase 28 oct 2026):
- Cada gravímetro tiene **una sola ecuación de deriva**, aunque haya datos de días diferentes.
- En el gravímetro 0.0959 hay un **salto**: los datos forman dos rectas. Se usa la que tiene **más datos** (ecuación 2).

Las lecturas vienen en grupos de tres; se usa el promedio de cada grupo.
La corrección de cada estación es **C_der = −D · (t − t_apertura del ciclo)**.

In [8]:
# ==== Paso 6: deriva instrumental ====
ARCHIVO_DERIVA = ORIG / "DERIVA.xlsx"
PASO68 = PROC / "Paso6a8_Gravedad_Observada_Facatativa.xlsx"

# Convención de signo para la gravedad observada (CONFIRMAR CON EL PROFESOR)
#  1 → g = g_base + (L − L_base), igual que en AMARRE BASE.xlsx
# -1 → g = g_base − (L − L_base), como en la cartera modelo
SIGNO = 1

der = openpyxl.load_workbook(ARCHIVO_DERIVA)

def promedios(ws, col_hora, col_lect, fila_ini):
    # Promedia las lecturas de deriva en grupos de tres (hora decimal, lectura de dial)
    filas = [(r[col_hora], r[col_lect]) for r in ws.iter_rows(min_row=fila_ini, values_only=True)
             if hasattr(r[col_hora], "hour") and isinstance(r[col_lect], (int, float))]
    grupos = []
    for k in range(0, len(filas) - 2, 3):
        g = filas[k:k + 3]
        grupos.append((np.mean([h.hour + h.minute / 60 for h, _ in g]), np.mean([L for _, L in g])))
    return grupos

# Gravímetro 0.0959: hora en columna D, lectura en F, desde la fila 8
g1 = promedios(der.worksheets[0], 3, 5, 8)
corte = (min(L for _, L in g1) + max(L for _, L in g1)) / 2      # separa las dos rectas del salto
rectas = [[p for p in g1 if p[1] >= corte], [p for p in g1 if p[1] < corte]]
ec2 = max(rectas, key=len)                                          # recta con más datos (se usa)
ec1 = min(rectas, key=len)                                          # recta con menos datos (no se usa)
# Gravímetro 0.0433: hora en columna C, lectura en E, desde la fila 7
g2 = promedios(der.worksheets[1], 2, 4, 7)

def ajuste(puntos, constante):
    t = np.array([p[0] for p in puntos]); g = np.array([p[1] for p in puntos]) * constante

    D, b = np.polyfit(t, g, 1)
    r2 = np.corrcoef(t, g)[0, 1] ** 2
    return D, b, r2

D1, b1, r2_1 = ajuste(ec2, CONST_GRAV1)
D2, b2, r2_2 = ajuste(g2, CONST_GRAV2)
print(f"Gravímetro 0.0959: ecuación 2 con {len(ec2)} grupos (la otra recta tiene {len(ec1)})")
print(f"   D = {D1:.4f} mGal/h   R² = {r2_1:.3f}")
print(f"Gravímetro 0.0433: {len(g2)} grupos")
print(f"   D = {D2:.4f} mGal/h   R² = {r2_2:.3f}")

Gravímetro 0.0959: ecuación 2 con 9 grupos (la otra recta tiene 5)
   D = -0.0242 mGal/h   R² = 0.245
Gravímetro 0.0433: 36 grupos
   D = -0.0229 mGal/h   R² = 0.425


## Paso 7. Corrección por cierre de cada ciclo

1. **Lectura corregida** = lectura (mGal) + corrección por marea + corrección por deriva.
2. Aun así, GPS-3 no da exactamente lo mismo al abrir y al cerrar el ciclo. Esa diferencia (**error de cierre**) se reparte linealmente en el tiempo:
   **C_cierre = (W_apertura − W_cierre) / (t_cierre − t_apertura) · (t − t_apertura)**
3. **Control (tablero del profesor):** después de la corrección, GPS-3 debe dar exactamente lo mismo al abrir y al cerrar cada ciclo.

## Paso 8. Gravedad observada absoluta

**g_obs = g(GPS-3) + SIGNO · (lectura final − lectura final de GPS-3 en la apertura)**, con g(GPS-3) = 977384.73 mGal.

Todo queda con fórmulas de Excel en `Paso6a8_Gravedad_Observada_Facatativa.xlsx`, para revisarlo celda por celda.

In [10]:
# ==== Pasos 7 y 8: cierre y gravedad observada (más la hoja Deriva con gráficas) ====
from openpyxl.chart import ScatterChart, Reference, Series
from openpyxl.chart.trendline import Trendline

wb = openpyxl.load_workbook(PASO45)

# --- Parámetros nuevos en la Leyenda ---
ley = wb["Leyenda"]
ley["A15"], ley["B15"] = "g absoluta GPS-3 (mGal)", round(G_BASE, 3)
ley["A16"], ley["B16"] = "Signo de la gravedad observada", SIGNO
ley["C16"] = "1: g = base + (L − L_base), como el amarre; −1: como la cartera modelo. Confirmar con el profesor."
for ref in ("A15", "A16", "C16"): ley[ref].font = f()
for ref in ("B15", "B16"): ley[ref].fill, ley[ref].font = AMARILLO, f(color="0000FF")

# --- Hoja Deriva con los datos, las ecuaciones y las gráficas ---
wd = wb.create_sheet("Deriva", 2)
wd["A1"] = "Paso 6 – Deriva instrumental (una ecuación por gravímetro)"
wd["A1"].font = Font(name=FUENTE, size=13, bold=True)

def tabla(fila0, col0, titulo, puntos, constante):
    # Escribe hora, lectura y g relativa; devuelve la primera y la última fila de datos
    wd.cell(fila0, col0, titulo).font = f(bold=True)
    for j, texto in enumerate(["Hora prom. (h)", "Lectura prom. (dial)", "g relativa (mGal)"]):
        encabezado(wd.cell(fila0 + 1, col0 + j), texto)
    for k, (t, L) in enumerate(puntos):
        r = fila0 + 2 + k
        wd.cell(r, col0, round(t, 4)).font = f()
        wd.cell(r, col0 + 1, round(L, 3)).font = f()
        wd.cell(r, col0 + 2, f"={get_column_letter(col0 + 1)}{r}*{constante}").font = f()
        for j in range(3): wd.cell(r, col0 + j).border = BORDE
    return fila0 + 2, fila0 + 1 + len(puntos)


for col in range(1, 13): wd.column_dimensions[get_column_letter(col)].width = 14
a1, z1 = tabla(3, 1, "Gravímetro 0.0959 – Ecuación 2 (usada)", ec2, CONST_GRAV1)
a0, z0 = tabla(z1 + 3, 1, "Gravímetro 0.0959 – Ecuación 1 (no usada)", ec1, CONST_GRAV1)
a2, z2 = tabla(3, 5, "Gravímetro 0.0433", g2, CONST_GRAV2)

# Resultados de la regresión con fórmulas de Excel
wd["I3"] = "Resultados"; wd["I3"].font = f(bold=True)
for j, texto in enumerate(["Gravímetro", "Pendiente D (mGal/h)", "Intercepto (mGal)", "R²"]):
    encabezado(wd.cell(4, 9 + j), texto)
for r, (nombre, ini, fin, cx, cy) in enumerate([("0.0959", a1, z1, "A", "C"), ("0.0433", a2, z2, "E", "G")], start=5):
    wd[f"I{r}"] = nombre
    wd[f"J{r}"] = f"=SLOPE({cy}{ini}:{cy}{fin},{cx}{ini}:{cx}{fin})"
    wd[f"K{r}"] = f"=INTERCEPT({cy}{ini}:{cy}{fin},{cx}{ini}:{cx}{fin})"
    wd[f"L{r}"] = f"=RSQ({cy}{ini}:{cy}{fin},{cx}{ini}:{cx}{fin})"
    for c in "IJKL": wd[f"{c}{r}"].font = f(); wd[f"{c}{r}"].border = BORDE
    for c in "JK": wd[f"{c}{r}"].number_format = "0.0000"
    wd[f"L{r}"].number_format = "0.000"
CELDA_D = {1: "Deriva!$J$5", 2: "Deriva!$J$6"}

# Gráfica de la deriva del gravímetro 0.0959 (las dos rectas)
graf = ScatterChart(); graf.title = "Deriva gravímetro 0.0959"; graf.style = 13
graf.x_axis.title = "Hora (h)"; graf.y_axis.title = "g relativa (mGal)"
for nombre, ini, fin, tendencia in (("Ecuación 2 (usada)", a1, z1, True), ("Ecuación 1", a0, z0, False)):
    s = Series(Reference(wd, min_col=3, min_row=ini, max_row=fin),
               Reference(wd, min_col=1, min_row=ini, max_row=fin), title=nombre)
    s.marker.symbol = "circle"; s.graphicalProperties.line.noFill = True
    if tendencia: s.trendline = Trendline(trendlineType="linear", dispEq=True, dispRSqr=True)
    graf.series.append(s)
graf.width, graf.height = 18, 9
wd.add_chart(graf, "I9")

# Gráfica de la deriva del gravímetro 0.0433

graf2 = ScatterChart(); graf2.title = "Deriva gravímetro 0.0433"; graf2.style = 13
graf2.x_axis.title = "Hora (h)"; graf2.y_axis.title = "g relativa (mGal)"
s = Series(Reference(wd, min_col=7, min_row=a2, max_row=z2),
           Reference(wd, min_col=5, min_row=a2, max_row=z2), title="Datos")
s.marker.symbol = "circle"; s.graphicalProperties.line.noFill = True
s.trendline = Trendline(trendlineType="linear", dispEq=True, dispRSqr=True)
graf2.series.append(s); graf2.width, graf2.height = 18, 9
wd.add_chart(graf2, "I29")

# --- Pasos 6, 7 y 8 en cada ciclo ---
nuevas = {"V": "Corrección deriva (mGal)", "W": "Lectura corregida marea + deriva (mGal)",
          "X": "Corrección cierre (mGal)", "Y": "Lectura final (mGal)", "Z": "g observada (mGal)"}
fin_ciclo = {}
for ws in wb.worksheets:
    if not ws.title.startswith("Ciclo"):
        continue
    n = int(ws.title.split()[1])
    grav = 1 if n <= 7 else 2
    for col, texto in nuevas.items():
        encabezado(ws[f"{col}5"], texto); ws.column_dimensions[col].width = 15
    ini = 6; fin = ini
    while ws[f"D{fin + 1}"].value is not None:
        fin += 1
    for r in range(ini, fin + 1):
        ws[f"V{r}"] = f"=-{CELDA_D[grav]}*(H{r}-$H${ini})"                                  # Paso 6
        ws[f"W{r}"] = f"=R{r}+U{r}+V{r}"                                                     # Paso 7.1
        ws[f"X{r}"] = f"=($W${ini}-$W${fin})/($H${fin}-$H${ini})*(H{r}-$H${ini})"            # Paso 7.2
        ws[f"Y{r}"] = f"=W{r}+X{r}"
        ws[f"Z{r}"] = f"=Leyenda!$B$15+Leyenda!$B$16*(Y{r}-$Y${ini})"                        # Paso 8
        for col in "VWXY": ws[f"{col}{r}"].number_format = "0.0000"
        ws[f"Z{r}"].number_format = "0.000"
    fin_ciclo[n] = (ini, fin)


# --- Control de cierre en el Resumen ---
res = wb["Resumen"]
for j, texto in ((14, "Error de cierre (mGal)"), (15, "Control: GPS-3 cierre − apertura (debe ser 0)")):
    encabezado(res.cell(3, j), texto); res.column_dimensions[get_column_letter(j)].width = 16
for n, (ini, fin) in fin_ciclo.items():
    q = f"'Ciclo {n}'!"
    res.cell(3 + n, 14, f"={q}W{fin}-{q}W{ini}").number_format = "0.0000"
    res.cell(3 + n, 15, f"=ROUND({q}Y{fin}-{q}Y{ini},6)").number_format = "0.000000"

wb.save(PASO68)
print("Guardado:", PASO68)

Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso6a8_Gravedad_Observada_Facatativa.xlsx


## Paso 9. Gravedad teórica

Fórmula de Somigliana con constantes WGS84, consistente con el datum que pide el taller:

γ = 978032.53359 · (1 + 0.00193185265241 · sen²φ) / √(1 − 0.00669437999013 · sen²φ)

## Paso 10. Aire libre, placa de Bouguer y Anomalía de Bouguer Simple

- Corrección de aire libre: C_AL = 0.3086 · h
- Corrección de placa de Bouguer: C_B = 0.04193 · ρ · h, con ρ = 2.67 g/cm³
- Anomalía de aire libre: AAL = g_obs − γ + C_AL
- Anomalía de Bouguer Simple: ABS = AAL − C_B

h es la altura elipsoidal en metros.

In [11]:
# ==== Pasos 9 y 10: gravedad teórica, aire libre, placa de Bouguer y Bouguer Simple ====
PASO910 = PROC / "Paso9y10_Bouguer_Simple_Facatativa.xlsx"
DENSIDAD = 2.67          # g/cm³, densidad de Bouguer acordada

wb = openpyxl.load_workbook(PASO68)

# Densidad como parámetro editable en la Leyenda
ley = wb["Leyenda"]
ley["A17"], ley["B17"] = "Densidad de Bouguer (g/cm³)", DENSIDAD
ley["A17"].font = f(); ley["B17"].fill, ley["B17"].font = AMARILLO, f(color="0000FF")

nuevas = {"AA": "Gravedad teórica γ (mGal)", "AB": "Corrección aire libre (mGal)",
          "AC": "Corrección placa de Bouguer (mGal)", "AD": "Anomalía aire libre (mGal)",
          "AE": "Anomalía de Bouguer Simple (mGal)"}

for ws in wb.worksheets:
    if not ws.title.startswith("Ciclo"):
        continue
    for col, texto in nuevas.items():
        encabezado(ws[f"{col}5"], texto); ws.column_dimensions[col].width = 15
    r = 6
    while ws[f"D{r}"].value is not None:
        # Paso 9: Somigliana con constantes WGS84 (latitud en la columna D)
        ws[f"AA{r}"] = (f"=978032.53359*(1+0.00193185265241*SIN(RADIANS(D{r}))^2)"
                        f"/SQRT(1-0.00669437999013*SIN(RADIANS(D{r}))^2)")
        # Paso 10: correcciones y anomalías (altura en la columna G)
        ws[f"AB{r}"] = f"=0.3086*G{r}"
        ws[f"AC{r}"] = f"=0.04193*Leyenda!$B$17*G{r}"
        ws[f"AD{r}"] = f"=Z{r}-AA{r}+AB{r}"
        ws[f"AE{r}"] = f"=AD{r}-AC{r}"
        for col in nuevas: ws[f"{col}{r}"].number_format = "0.000"
        r += 1


wb.save(PASO910)
print("Guardado:", PASO910)

Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\Paso9y10_Bouguer_Simple_Facatativa.xlsx


## Paso 11. Archivo para el compañero

CSV con una fila por estación (la base GPS-3 una sola vez): ID, coordenadas, altura, g_obs, γ, correcciones y anomalías.
Con este archivo el compañero calcula la corrección topográfica y la Anomalía de Bouguer Total.

In [12]:
# ==== Paso 11: CSV para el compañero (una fila por estación) ====
import pandas as pd

def somigliana(lat):
    s2 = np.sin(np.radians(lat)) ** 2
    return 978032.53359 * (1 + 0.00193185265241 * s2) / np.sqrt(1 - 0.00669437999013 * s2)

wb = openpyxl.load_workbook(PASO45)      # aquí ya están las correcciones por marea (columna T)
filas = []
for ws in wb.worksheets:
    if not ws.title.startswith("Ciclo"):
        continue
    n = int(ws.title.split()[1])
    D = D1 if n <= 7 else D2
    const = CONST_GRAV1 if n <= 7 else CONST_GRAV2
    datos = []
    r = 6
    while ws[f"D{r}"].value is not None:
        datos.append(dict(ID=ws[f"B{r}"].value, Estacion=ws[f"C{r}"].value, Ciclo=n,
                          Latitud=ws[f"D{r}"].value, Longitud=-ws[f"E{r}"].value, h=ws[f"G{r}"].value,
                          Este=ws[f"P{r}"].value, Norte=ws[f"Q{r}"].value,
                          t=ws[f"H{r}"].value, L=ws[f"K{r}"].value * const, marea=-ws[f"T{r}"].value))
        r += 1
    # Mismo cálculo que las fórmulas del Excel (pasos 6, 7 y 8)
    t0, tf = datos[0]["t"], datos[-1]["t"]
    W = [d["L"] + d["marea"] - D * (d["t"] - t0) for d in datos]
    Y = [w + (W[0] - W[-1]) / (tf - t0) * (d["t"] - t0) for w, d in zip(W, datos)]
    for d, y in zip(datos, Y):
        d["g_obs"] = round(G_BASE, 3) + SIGNO * (y - Y[0])
    filas += datos

df = pd.DataFrame(filas)

df["gamma"] = somigliana(df["Latitud"])
df["C_AL"] = 0.3086 * df["h"]
df["C_B"] = 0.04193 * DENSIDAD * df["h"]
df["AAL"] = df["g_obs"] - df["gamma"] + df["C_AL"]
df["ABS"] = df["AAL"] - df["C_B"]

# Una sola fila para la base GPS-3; el resto son las estaciones
base = df[df["Estacion"] == BASE].iloc[[0]].assign(ID=BASE)
estaciones = df[df["Estacion"] != BASE]
salida = pd.concat([base, estaciones])[["ID", "Estacion", "Ciclo", "Este", "Norte", "h", "Latitud", "Longitud",
                                         "g_obs", "gamma", "C_AL", "C_B", "AAL", "ABS"]].round(4)

CSV = PROC / "ABS_para_companero.csv"
salida.to_csv(CSV, index=False, encoding="utf-8-sig")
print(f"Guardado: {CSV}  ({len(salida)} puntos)")
print(f"ABS: mínimo {salida['ABS'].min():.2f} mGal, máximo {salida['ABS'].max():.2f} mGal")
salida.head()

Guardado: c:\Users\Daniel\Documents\2026-3\geofisica\Taller 2\Gravimetria-Facatativa\datos\procesados\ABS_para_companero.csv  (98 puntos)
ABS: mínimo -169.66 mGal, máximo -164.81 mGal


,ID,Estacion,Ciclo,Este,Norte,h,Latitud,Longitud,g_obs,gamma,C_AL,C_B,AAL,ABS
0,GPS-3,GPS-3,1,974495.614,1025740.326,2617.994,4.8289,-74.3074,977384.7270,978069.1226,807.9129,293.0925,123.5173,-169.5752
1,C01-STK502,STK502,1,974505.103,1025552.946,2617.646,4.8272,-74.3073,977384.8291,978069.0970,807.8056,293.0536,123.5377,-169.5159
2,C01-SP510,SP510,1,974514.661,1025570.759,2617.635,4.8274,-74.3073,977384.8534,978069.0994,807.8022,293.0524,123.5561,-169.4962
3,C01-SP518,SP518,1,974523.911,1025588.265,2617.524,4.8276,-74.3072,977384.9654,978069.1018,807.7679,293.0399,123.6315,-169.4085
4,C01-SP526,SP526,1,974531.312,1025606.693,2617.440,4.8277,-74.3071,977384.8958,978069.1043,807.7420,293.0305,123.5335,-169.4970
